In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, TensorDataset, random_split
import sys
import os

# Adds folder to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from Utils.utils import get_absorptance, si_eps, get_sine_eps, get_weighted_absorptance

sim_dtype = torch.complex64
geo_dtype = torch.float32

In [ ]:
Loaded = torch.load("../Data/Modulated_Silicon_Data_5000_structures.pt")

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, output_size, hidden_layers, hidden_nodes, dropout_prob=0.05):
        super(MLP, self).__init__()
        
        # Store dropout probability
        self.dropout_prob = dropout_prob
        layers = []
        
        # First Hidden Layer
        layers.append(torch.nn.Linear(input_size, hidden_nodes))
        layers.append(torch.nn.BatchNorm1d(hidden_nodes))
        layers.append(torch.nn.ReLU())
        # Dropout to prevent overfitting
        layers.append(torch.nn.Dropout(self.dropout_prob))
        
        # Subsequent Hidden Layers
        for _ in range(hidden_layers - 1):
            layers.append(torch.nn.Linear(hidden_nodes, hidden_nodes))
            layers.append(torch.nn.BatchNorm1d(hidden_nodes))
            layers.append(torch.nn.ReLU())
            layers.append(torch.nn.Dropout(self.dropout_prob))
            
        #Output Layer
        layers.append(torch.nn.Linear(hidden_nodes, output_size))
        
        self.network = torch.nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() == 3:
            # Polar-to-Cartesian Feature Transformation
            amplitudes = x[:, :, 0]
            phases = x[:, :, 1]
            an = amplitudes * torch.cos(phases)
            bn = amplitudes * torch.sin(phases)
            x = torch.cat((an, bn),dim=1)
        return self.network(x)

In [ ]:
class ConvolutionalRegression(torch.nn.Module):
    def __init__(self, output_size, conv_layers, conv_channels, kernel_size, stride, 
                 hidden_layers, hidden_nodes, dropout_prob=0.05,nx=5000):
        super(ConvolutionalRegression, self).__init__()
        
        # Store dropout probability
        self.dropout_prob = dropout_prob
        layers = []
        
        # Convolutional Layers
        in_channels = 1  # Assuming single channel input
        for _ in range(conv_layers):
            layers.append(torch.nn.Conv1d(in_channels, conv_channels, kernel_size, stride=stride, padding=kernel_size//2))
            layers.append(torch.nn.BatchNorm1d(conv_channels))
            layers.append(torch.nn.ReLU())
            layers.append(torch.nn.Dropout(self.dropout_prob))
            in_channels = conv_channels
        
        self.conv_network = torch.nn.Sequential(*layers)
        
        # Calculate the size after convolution to define the first linear layer
        conv_output_size = conv_channels  # After global average pooling
        
        # Fully Connected Layers
        fc_layers = []
        
        # First Hidden Layer
        fc_layers.append(torch.nn.Linear(conv_output_size, hidden_nodes))
        fc_layers.append(torch.nn.BatchNorm1d(hidden_nodes))
        fc_layers.append(torch.nn.ReLU())
        fc_layers.append(torch.nn.Dropout(self.dropout_prob))
        
        # Subsequent Hidden Layers
        for _ in range(hidden_layers - 1):
            fc_layers.append(torch.nn.Linear(hidden_nodes, hidden_nodes))
            fc_layers.append(torch.nn.BatchNorm1d(hidden_nodes))
            fc_layers.append(torch.nn.ReLU())
            fc_layers.append(torch.nn.Dropout(self.dropout_prob))
            
        # Output Layer
        fc_layers.append(torch.nn.Linear(hidden_nodes, output_size))
        
        self.fc_network = torch.nn.Sequential(*fc_layers)

    def forward(self, x):
        if x.dim() == 3:
            # Polar-to-Cartesian Feature Transformation
            amplitudes = x[:, :, 0]
            phases = x[:, :, 1]
            an = amplitudes * torch.cos(phases)
            bn = amplitudes * torch.sin(phases)
            x = torch.cat((an, bn),dim=1)

        # Fourier Feature Transformation (for the convolutional layers to scan the grating)
        cosines = torch.cos(2.*np.pi*torch.arange(1,x.shape[1]//2+1,device=x.device).unsqueeze(1)*torch.linspace(0,1000,5000,device=x.device).unsqueeze(0)/1000.)
        sines = torch.sin(2.*np.pi*torch.arange(1,x.shape[1]//2+1,device=x.device).unsqueeze(1)*torch.linspace(0,1000,5000,device=x.device).unsqueeze(0)/1000.)
        cosines = x[:,:x.shape[1]//2] @ cosines
        sines = x[:,x.shape[1]//2:] @ sines
        x = cosines+sines
        
        # Add channel dimension for convolution
        x = x.unsqueeze(1)  # Shape: (batch_size, 1, features)
        
        # Pass through convolutional layers
        x = self.conv_network(x)
        x = torch.mean(x, dim=2)  # Global average pooling
        
        # Pass through fully connected layers
        x = self.fc_network(x)
        
        return x

In [ ]:
#Make a simple dataloader to check the data
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = TensorDataset(Loaded['features'].to(device), Loaded['targets'].to(device))
total_dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
train_size = int(0.80 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
#initialize the MLP
input_size = 10  # 5 amplitudes + 5 phases
output_size = 2  # Number of absorptance values to predict
hidden_layers = 6
hidden_nodes = 256
model = MLP(input_size, output_size, hidden_layers, hidden_nodes)
criterion = torch.nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4,weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=100)

# convert to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f'Total number of parameters in MLP: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

#Create DataLoaders
BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

num_epochs = 5000

#setup losses
train_loss_history = []
val_loss_history = []

best_val_loss = float('inf') # For early stopping
patience = 1000              # How many epochs to wait for improvement
patience_counter = 0

pbar = tqdm(range(num_epochs), desc="Training")

for epoch in pbar:
    # Training
    model.train()
    total_train_loss = 0.0
    
    for inputs, targets in train_loader:

        inputs, targets = inputs, targets
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        # Accumulate the batch loss (mean loss per batch)
        total_train_loss += loss.item()
    
    # Calculate average training loss for the epoch
    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)

    # Validation
    model.eval() # Set model to evaluation mode
    total_val_loss = 0.0
    
    with torch.no_grad():
        for inputs, targets in val_loader: # Use val_loader for validation loss

            inputs, targets = inputs, targets
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_val_loss += loss.item()
    
    # Calculate average validation loss for the epoch
    avg_val_loss = total_val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)
    scheduler.step(avg_val_loss)

    # Update Pbar and check for early stopping
    pbar.set_description(
        f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4e} | Val Loss: {avg_val_loss:.4e} | lr: {optimizer.param_groups[0]['lr']:.4e}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Potentially save the best model
        best_weights = model.state_dict()
        # torch.save(model.state_dict(), 'best_mlp_model.pt')
        
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Validation loss did not improve for {patience} epochs. Stopping early at epoch {epoch+1}.")
            # Optionally save the model here
            # torch.save(model.state_dict(), 'early_stopped_mlp_model.pt')
            print(f'loading best weights from epoch with val loss {best_val_loss}')
            model.load_state_dict(best_weights)
            break
            

In [ ]:
plt.plot(train_loss_history, label='Train Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Average L1 Error')
plt.yscale('log')
plt.title(f'Training and Validation Loss over Epochs (MLP). Number of total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
plt.legend()
plt.show()

In [ ]:
#initialize the Convolutional Regression Model
output_size = 2  # Number of absorptance values to predict
hidden_layers = 5
hidden_nodes = 128
conv_layers = 5
conv_channels = 32
kernel_size = 64
stride = 16
model_conv = ConvolutionalRegression(output_size, conv_layers, conv_channels, kernel_size, stride, hidden_layers, hidden_nodes, nx=500) 
criterion = torch.nn.L1Loss()
optimizer = torch.optim.AdamW(model_conv.parameters(), lr=1e-3,weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=100)

# convert to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model_conv.to(device)
print(f'Total number of parameters: {sum(p.numel() for p in model_conv.parameters() if p.requires_grad)}')
pass

#Create DataLoaders
BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

num_epochs = 5000

#setup losses
train_loss_history_conv = []
val_loss_history_conv = []

best_val_loss = float('inf') # For early stopping
patience = 1000               # How many epochs to wait for improvement
patience_counter = 0

pbar = tqdm(range(num_epochs), desc="Training")

for epoch in pbar:
    # Training
    model.train()
    total_train_loss = 0.0
    
    for inputs, targets in train_loader:

        inputs, targets = inputs, targets
        optimizer.zero_grad()
        outputs = model_conv(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        # Accumulate the batch loss (mean loss per batch)
        total_train_loss += loss.item()
    
    # Calculate average training loss for the epoch
    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_history_conv.append(avg_train_loss)

    # Validation
    model_conv.eval() # Set model to evaluation mode
    total_val_loss = 0.0
    
    with torch.no_grad():
        for inputs, targets in val_loader: # Use val_loader for validation loss

            inputs, targets = inputs, targets
            outputs = model_conv(inputs)
            loss = criterion(outputs, targets)
            total_val_loss += loss.item()
    
    # Calculate average validation loss for the epoch
    avg_val_loss = total_val_loss / len(val_loader)
    val_loss_history_conv.append(avg_val_loss)
    scheduler.step(avg_val_loss)

    # Update Pbar and check for early stopping
    pbar.set_description(
        f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4e} | Val Loss: {avg_val_loss:.4e} | lr: {optimizer.param_groups[0]['lr']:.4e}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Potentially save the best model
        best_weights = model_conv.state_dict()
        # torch.save(model.state_dict(), 'best_conv_model.pt')
        
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Validation loss did not improve for {patience} epochs. Stopping early at epoch {epoch+1}.")
            # Optionally save the model here
            # torch.save(model.state_dict(), 'early_stopped_conv_model.pt')
            print(f'loading best weights from epoch with val loss {best_val_loss}')
            model_conv.load_state_dict(best_weights)
            break
            

In [ ]:
plt.plot(train_loss_history, label='Train Loss MLP')
plt.plot(val_loss_history, label='Validation Loss MLP')
plt.plot(train_loss_history_conv, label='Train Loss Conv')
plt.plot(val_loss_history_conv, label='Validation Loss Conv')
plt.xlabel('Epoch')
plt.ylabel('Average L1 Error')
plt.yscale('log')
plt.title(f'Training and Validation Loss over Epochs (Convolutional Regression).\nNumber of total parameters Conv: {sum(p.numel() for p in model_conv.parameters() if p.requires_grad)}\nNumber of total parameters MLP: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
plt.legend()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
plot_features = torch.cat([batch[0] for batch in val_loader], dim=0)
plot_targets  = torch.cat([batch[1] for batch in val_loader], dim=0)
model_conv.eval()
model.eval()
axs[0].scatter(plot_targets[:,0].cpu(), plot_targets[:,1].cpu(), label='True Values')
axs[1].scatter(plot_targets[:,0].cpu(), plot_targets[:,1].cpu(), label='True Values', alpha=0.5,c='g')
axs[1].scatter(model(plot_features).detach().cpu()[:,0], model(plot_features).detach().cpu()[:,1], label='MLP Predictions')
axs[2].scatter(plot_targets[:,0].cpu(), plot_targets[:,1].cpu(), label='True Values', alpha=0.5,c='g')
axs[2].scatter(model_conv(plot_features).detach().cpu()[:,0], model_conv(plot_features).detach().cpu()[:,1], label='Conv Predictions')
axs[0].set_xlabel('Absorptance energy spectrum')
axs[1].set_xlabel('Absorptance energy spectrum')
axs[2].set_xlabel('Absorptance energy spectrum')
axs[0].set_ylabel('Absorptance photon spectrum')
axs[1].set_ylabel('Absorptance photon spectrum')
axs[2].set_ylabel('Absorptance photon spectrum')
axs[0].set_title('True Values')
axs[1].set_title('MLP Predictions')
axs[2].set_title('Conv Predictions')
axs[0].legend()
axs[1].legend()
axs[2].legend()
# Use True data to set common axes limits
true_x = Loaded['targets'][:,0].cpu().numpy()
true_y = Loaded['targets'][:,1].cpu().numpy()
x_min = true_x.min()*0.98
x_max = true_x.max()*1.02
y_min = true_y.min()*0.98
y_max = true_y.max()*1.02
axs[0].set_xlim([x_min, x_max])
axs[0].set_ylim([y_min, y_max])
axs[1].set_xlim([x_min, x_max])
axs[1].set_ylim([y_min, y_max])
axs[2].set_xlim([x_min, x_max])
axs[2].set_ylim([y_min, y_max])
plt.show()

In [ ]:
n_osc = 4
test = torch.zeros((1,n_osc,2)).to(device)
test[0,:,0] = torch.rand(n_osc)*100/n_osc
test[0,:,1] = torch.rand(n_osc)*2.*np.pi
get_weighted_absorptance(test[0])

In [ ]:
model_conv.eval()
model_conv(test)

## Testing

In [ ]:
test_param = torch.tensor([[20,0],[20,np.pi],[20,0]],dtype=geo_dtype,device=device)

In [ ]:
argmax = torch.argmax(Loaded['targets'][:,1])
best_param = Loaded['features'][argmax].to(device)

In [ ]:
get_weighted_absorptance(test_param)

In [ ]:
get_weighted_absorptance(best_param)

In [ ]:
%timeit get_absorptance(test_param)

In [ ]:
sine_eps = get_sine_eps(torch.linspace(0,1000,1000,device=device),best_param,1000,si_eps(500))

In [ ]:
difference = torch.abs(model(Loaded['features'].to(device)) - Loaded['targets'].to(device))
difference_conv = torch.abs(model_conv(Loaded['features'].to(device)) - Loaded['targets'].to(device))
relative_difference = difference / Loaded['targets'].to(device)
percent_difference = relative_difference * 100
relative_difference_conv = difference_conv / Loaded['targets'].to(device)
percent_difference_conv = relative_difference_conv * 100

In [ ]:
difference.max()

In [ ]:
difference_conv.max()

In [ ]:
outputs = model(Loaded['features'].to(device))

In [ ]:
def objective_function(params):
    """
    Objective function to minimize: negative weighted absorptance.
    """
    transformed = torch.stack((torch.exp(params[:,0]), 2*torch.atan(params[:,1])), dim=-1)
    weighted_absorptance = get_weighted_absorptance(transformed)
    return -weighted_absorptance[0] - weighted_absorptance[1]

In [ ]:
#find optimal parameters
init_params = torch.tensor([[1,0] for i in range(100)],dtype=geo_dtype,device=device,requires_grad=True)
opt = torch.optim.AdamW([init_params], lr=1,weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.99)
max_scheduler_steps = 200

pbar = tqdm(range(500), desc="Optimizing Parameters")
best_loss = float('inf')

for it in pbar:
    opt.zero_grad()
    loss = objective_function(init_params)
    loss.backward()
    opt.step()
    if it < max_scheduler_steps:
        scheduler.step()
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_params = init_params.detach().clone()
    pbar.set_description(f"Iteration {it}: Loss = {loss.item():.4f} - lr = {scheduler.get_last_lr()[0]:.4f}")

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
del loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
transformed = torch.stack((torch.exp(best_params[:,0]), 2*torch.atan(best_params[:,1])), dim=-1).detach()

In [ ]:
transformed

In [ ]:
test_param = Loaded['features'][argmax].to(device)

In [ ]:
Loaded['targets'][argmax]

In [ ]:
get_weighted_absorptance(transformed)

In [ ]:
random_params = torch.zeros((3,2)).to(device)
random_params[:,0] = torch.rand(3)*20
random_params[:,1] = torch.rand(3)*2*np.pi - np.pi

In [ ]:
sine_eps_test = get_sine_eps(torch.linspace(0,1000,1000,device=device),test_param,1000,si_eps(500))

In [ ]:
sine_eps_gradient = get_sine_eps(torch.linspace(0,1000,1000,device=device),transformed,1000,si_eps(500))

In [ ]:
sine_eps_random = get_sine_eps(torch.linspace(0,1000,1000,device=device),random_params,1000,si_eps(500))

In [ ]:
get_weighted_absorptance(transformed)

In [ ]:
plt.plot(sine_eps_random.real.cpu().numpy(),label='Real')

In [ ]:
plt.plot(sine_eps_gradient.real.cpu().numpy(),label='Real')